In [ ]:
import numpy as np
import pandas as pd
from math import factorial

def permutation_entropy(signal, d, t):
    """
    Calculates permutation entropy of the response variable, which is a measure of its complexity,
    based on the ordinal patterns of its values.

    Args:
        signal (np.ndarray): The response variable.
        d (int): Embedding dimension. The length of the patterns to consider.
        t (int): Time delay. The number of time steps between elements in a pattern.

    Returns:
        float: The normalized permutation entropy, ranging from 0 to 1.
    """
    n = len(signal)

    # Input validation
    if d > n:
        raise ValueError("The embedding dimension (d) cannot be greater than the signal length (n).")
    if t < 1 or not isinstance(t, int):
        raise ValueError("The time delay (t) must be a positive integer.")

    # Create embedded vectors based on the embedding dimension and time delay
    # The list comprehension creates subsequences of length `d` with a step of `t`
    # for each possible starting point in the signal.
    embedded_vectors = [signal[i:i + d * t:t] for i in range(n - (d - 1) * t)]

    # Determine the ordinal pattern (permutation) for each embedded vector
    # np.argsort returns the indices that would sort the array, which defines the permutation.
    permutations = [tuple(np.argsort(v)) for v in embedded_vectors]

    # Count the occurrences of each unique permutation
    counts = {}
    for p in permutations:
        counts[p] = counts.get(p, 0) + 1

    # Calculate the probability of each permutation
    probs = np.array(list(counts.values())) / len(permutations)

    # Calculate Shannon entropy from the probabilities
    entropy = -np.sum(probs * np.log(probs))

    # Normalize the entropy by the maximum possible entropy
    # The maximum entropy for dimension `d` is log(d!)
    max_entropy = np.log(factorial(d))

    return entropy / max_entropy

# --- Main Execution ---

# Parameters
d_values = [3, 4, 5] # The choice of d should satisy the inequality N >= 5d!
t_values = [1, 2]

# Load data from the CSV file
#df = pd.read_csv('Spline_Simulations.csv') #the IC data set
df = pd.read_csv('Spline_Simulations_OOC_New_3.csv') # One of the OOC data sets
# Get simulation columns
simulation_cols = [col for col in df.columns if col.startswith("Simulation_")]

print("Calculating and saving permutation entropy for each combination of d and t to a separate file...")

# Loop through each combination of d and t
for d in d_values:
    for t in t_values:
        print(f"  Processing with d={d}, t={t}...")
        
        # Calculate a single PE value for each entire simulation
        single_pe_values = {}
        for sim_col in simulation_cols:
            pe_value = permutation_entropy(df[sim_col].values, d, t)
            single_pe_values[sim_col] = pe_value
        
        # Create the output DataFrame with only the simulation and H_value columns
        output_df = pd.DataFrame(single_pe_values.items(), columns=['Simulation', 'H_value'])
        
        # Generate the unique filename
        #filename = f"IC_H_values_Spline_Simulations_d{d}_t{t}.csv"
        filename = f"OOC_H_values_Spline_Simulations_New_3_d{d}_t{t}.csv"
        
        # Save the results to a new CSV file
        output_df.to_csv(filename, index=False)
        print(f"  Results for d={d}, t={t} saved to {filename}")

print("\nAll calculations and file saving complete.")